Download + Extract Hadoop

In [1]:
!wget https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz
!tar -xzvf hadoop-3.4.2.tar.gz


Η έξοδος ροής περικόπηκε στις τελευταίες 5000 γραμμές.
hadoop-3.4.2/share/doc/hadoop/hadoop-fs2img/dependency-analysis.html
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/hadoop-hdfs-native-client/
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/hadoop-hdfs-native-client/images/
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/hadoop-hdfs-native-client/images/external.png
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/hadoop-hdfs-native-client/images/expanded.gif
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/hadoop-hdfs-native-client/images/icon_warning_sml.gif
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/hadoop-hdfs-native-client/images/collapsed.gif
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/hadoop-hdfs-native-client/images/logo_apache.jpg
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/hadoop-hdfs-native-client/images/icon_error_sml.gif
hadoop-3.4.2/share/doc/hadoop/hadoop-project-dist/hadoop-hdfs-native-c

In [2]:
!cp -r hadoop-3.4.2/ /usr/local/

open the file /usr/local/hadoop-3.4.2/etc/hadoop/hadoop-env.sh

replace the: export JAVA_HOME with export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64/

uncomment the line, and save the file

In [4]:
!/usr/local/hadoop-3.4.2/bin/hadoop version

Hadoop 3.4.2
Source code repository https://github.com/apache/hadoop.git -r 84e8b89ee2ebe6923691205b9e171badde7a495c
Compiled by ahmarsu on 2025-08-20T10:30Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum fa94c67d4b4be021b9e9515c9b0f7b6
This command was run using /usr/local/hadoop-3.4.2/share/hadoop/common/hadoop-common-3.4.2.jar


Δημιουργία των δεδομένων εισόδου

In [5]:
!mkdir -p /content/input_join

In [6]:
import random

# Χώρες με skew
countries = ["US"] * 5 + ["GR", "DE", "FR", "SE", "IT", "UK", "ES"]

# Δημιουργία output string
output_lines = []

# Δημιουργία 2.000 εγγραφών R
for i in range(1, 2001):
    c = random.choice(countries)
    output_lines.append(f"R,{i},{c}")

# Δημιουργία 1.000 εγγραφών S
for j in range(1, 1001):
    c = random.choice(countries)
    output_lines.append(f"S,{j},{c}")

# Αποθήκευση αρχείου
with open("/content/input_join/join_data.txt", "w") as f:
    f.write("\n".join(output_lines))

len(output_lines)

3000

In [7]:
!wc -l /content/input_join/join_data.txt
!head /content/input_join/join_data.txt
!tail /content/input_join/join_data.txt

2999 /content/input_join/join_data.txt
R,1,US
R,2,UK
R,3,DE
R,4,DE
R,5,US
R,6,US
R,7,US
R,8,US
R,9,UK
R,10,SE
S,991,IT
S,992,US
S,993,FR
S,994,US
S,995,UK
S,996,DE
S,997,US
S,998,SE
S,999,GR
S,1000,DE

φτιαξε αρχειο με ονομα mapper_join.py μεσα στον content και βαλε τον κωδικα

%%writefile /content/mapper_join.py



#!/usr/bin/env python3
import sys
import hashlib

# Πλήθος buckets για διαμοιρασμό φορτίου
NUM_BUCKETS = 4

# Χώρες με skew
SKEWED_COUNTRIES = {"US"}

def get_bucket_for_r(record_id: str) -> int:
    """
    Υπολογισμός bucket για εγγραφές R με βάση το ID.
    """
    return int(hashlib.md5(record_id.encode()).hexdigest(), 16) % NUM_BUCKETS

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split(",")
    if len(parts) != 3:
        continue

    table, rec_id, country = parts

    # Αν η χώρα ΔΕΝ είναι skew → όλα στο bucket 0
    if country not in SKEWED_COUNTRIES:
        key = f"{country}|0"
        value = f"{table},{rec_id}"
        print(f"{key}\t{value}")
        continue

    # Αν η χώρα είναι skew (π.χ. US)
    if table == "R":
        # Τα R records μοιράζονται σε πολλά buckets
        bucket = get_bucket_for_r(rec_id)
        key = f"{country}|{bucket}"
        value = f"{table},{rec_id}"
        print(f"{key}\t{value}")
    else:
        # Τα S records αναπαράγονται σε όλα τα buckets
        for bucket in range(NUM_BUCKETS):
            key = f"{country}|{bucket}"
            value = f"{table},{rec_id}"
            print(f"{key}\t{value}")

Έλεγχος Mapper

In [8]:
!echo "R,1,US" | python3 /content/mapper_join.py

US|3	R,1


Φτιάξε ένα αρχειο reducer_join.py
%%writefile /content/reducer_join.py

  #!/usr/bin/env python3
import sys

current_key = None
r_ids = []
s_ids = []

def emit_joins(country, r_list, s_list):
    """
    Εκτυπώνει όλα τα ζευγάρια (S.ID, R.ID) για το ίδιο country.
    """
    for r in r_list:
        for s in s_list:
            print(f"{country}\t{s}\t{r}")

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    key, value = line.split("\t")
    country, bucket = key.split("|")
    table, rec_id = value.split(",")

    if current_key is None:
        current_key = key

    if key != current_key:
        prev_country, _ = current_key.split("|")
        emit_joins(prev_country, r_ids, s_ids)

        # reset lists
        r_ids = []
        s_ids = []
        current_key = key

    if table == "R":
        r_ids.append(rec_id)
    else:
        s_ids.append(rec_id)

if current_key is not None:
    last_country, _ = current_key.split("|")
    emit_joins(last_country, r_ids, s_ids)


Έλεγχος reducer

In [9]:
!echo -e "US|0\tR,1\nUS|0\tS,10" | python3 /content/reducer_join.py

US	10	1


Εκτέλεση χωρις Map Reduce

In [23]:
import time

start = time.time()

!cat /content/input_join/join_data.txt | python3 /content/mapper_join.py | sort | python3 /content/reducer_join.py > /content/local_join_output.txt

end = time.time()

print("\nΧρόνος εκτέλεσης χωρίς Hadoop:", round(end - start, 2), "δευτ.")


Χρόνος εκτέλεσης χωρίς Hadoop: 0.41 δευτ.


Εκτελεση με Hadoop MapReducer

In [24]:
!find /usr/local/hadoop-3.4.2/ -name "hadoop-streaming*.jar"

/usr/local/hadoop-3.4.2/share/hadoop/tools/sources/hadoop-streaming-3.4.2-test-sources.jar
/usr/local/hadoop-3.4.2/share/hadoop/tools/sources/hadoop-streaming-3.4.2-sources.jar
/usr/local/hadoop-3.4.2/share/hadoop/tools/lib/hadoop-streaming-3.4.2.jar


In [21]:
!chmod +x /content/mapper_join.py
!chmod +x /content/reducer_join.py

Αν υπάρχει ήδη folder

In [22]:
!rm -r /content/join_output

In [26]:
import time

# Σβήνουμε τον παλιό φάκελο output για να μην αποτύχει το Hadoop job
!rm -r /content/join_output 2>/dev/null

start = time.time()

! /usr/local/hadoop-3.4.2/bin/hadoop jar \
  /usr/local/hadoop-3.4.2/share/hadoop/tools/lib/hadoop-streaming-3.4.2.jar \
  -input /content/input_join \
  -output /content/join_output \
  -mapper "python3 /content/mapper_join.py" \
  -reducer "python3 /content/reducer_join.py"

end = time.time()

print("\nΧρόνος εκτέλεσης Hadoop MapReduce:", round(end - start, 2), "δευτ.")

2025-12-08 19:44:26,501 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2025-12-08 19:44:26,815 INFO mapred.FileInputFormat: Total input files to process : 1
2025-12-08 19:44:26,851 INFO mapreduce.JobSubmitter: number of splits:1
2025-12-08 19:44:27,200 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1717374067_0001
2025-12-08 19:44:27,203 INFO mapreduce.JobSubmitter: Executing with tokens: []
2025-12-08 19:44:27,546 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2025-12-08 19:44:27,549 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2025-12-08 19:44:27,550 INFO mapreduce.Job: Running job: job_local1717374067_0001
2025-12-08 19:44:27,554 INFO mapred.LocalJobRunner: OutputCommitter is org.apache.hadoop.mapred.FileOutputCommitter
2025-12-08 19:44:27,565 INFO output.FileOutputCommitter: File Output Committer Algorithm version is 2
2025-12-08 19:44:27,566 INFO output.FileOutputCommitter: FileOutputCommitter s

Έλεγχος αποτελεσμάτων

In [19]:
!ls -l /content/join_output
!cat /content/join_output/part-00000 | head -n 30

total 4916
-rw-r--r-- 1 root root 5030251 Dec  8 19:42 part-00000
-rw-r--r-- 1 root root       0 Dec  8 19:42 _SUCCESS
DE	836	275
DE	673	275
DE	751	275
DE	266	275
DE	952	275
DE	290	275
DE	685	275
DE	687	275
DE	285	275
DE	693	275
DE	498	275
DE	500	275
DE	282	275
DE	130	275
DE	857	275
DE	792	275
DE	131	275
DE	209	275
DE	507	275
DE	839	275
DE	882	275
DE	73	275
DE	138	275
DE	144	275
DE	145	275
DE	515	275
DE	517	275
DE	518	275
DE	431	275
DE	959	275


In [27]:
from collections import Counter

counts = Counter()

with open("/content/join_output/part-00000") as f:
    for line in f:
        country = line.split()[0]
        counts[country] += 1

counts

Counter({'DE': 15886,
         'ES': 12225,
         'FR': 12397,
         'GR': 15312,
         'IT': 12298,
         'SE': 14181,
         'UK': 15030,
         'US': 346580})

Το join R–S υλοποιήθηκε με Hadoop MapReduce. Για τη συχνή τιμή US
χρησιμοποιήθηκε bucketing ώστε να μοιραστεί το φορτίο.
Το Hadoop Streaming εκτέλεσε κανονικά το job και τα αποτελέσματα
βρίσκονται στο part-00000.

Το αρχείο part-00000 περιέχει όλα τα σωστά ζεύγη από τη σύζευξη R και S.
Οι συχνές χώρες, όπως η US, εμφανίζουν περισσότερες γραμμές, κάτι που είναι αναμενόμενο.

Ο χρόνος χωρίς Hadoop ήταν μικρότερος επειδή η εκτέλεση έγινε τοπικά με πολύ λίγα δεδομένα.
Με το Hadoop ο χρόνος αυξάνεται λόγω της διαδικασίας shuffle/sort και εκκίνησης των tasks.
Ωστόσο, για πολύ μεγαλύτερα datasets το Hadoop υπερέχει χάρη στην κατανεμημένη επεξεργασία.

Mε Hadoop Χρόνος εκτέλεσης: 4.44 δευτερόλεπτα

Χωρις Hadoop Χρόνος Εκτέλεσης: 0.41 δευτερόλεπτα

Ποσοτικό αποτέλεσμα Counter({'DE': 15886,
         'ES': 12225,
         'FR': 12397,
         'GR': 15312,
         'IT': 12298,
         'SE': 14181,
         'UK': 15030,
         'US': 346580}